## test new histograms locally

In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate

In [4]:
import coffea
print(coffea.__version__)
import inspect
from coffea import processor

print(inspect.getfile(processor.DaskExecutor))

2025.11.0
/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py


In [5]:
from coffea.processor import executor
import inspect

src = inspect.getsource(executor)

for i, line in enumerate(src.splitlines()):
    if "client.submit" in line:
        print(f"{i:4d}: {line}")

 727:                         items, repeat(self.client.submit(lambda x: x, self.heavy_input))


In [6]:
from coffea.processor import executor
import inspect

src = inspect.getsource(executor)

keywords = [
    "client.submit",
    "_work_function",
    "automatic_retries",
    "process",
    "submit(",
    "run_uproot_job",
]

for i, line in enumerate(src.splitlines()):
    for k in keywords:
        if k in line:
            print(f"{i:4d}: {line}")
            break

  35: from .processor import ProcessorABC
 390:                         FH.add_merge(pool.submit(merge_fcn, batch))
 400:             else:  # Merge within process
 523:             Number of parallel processes for futures (default 1)
 542:             Default is ``False`` - results get merged as they finish in the main process.
 550:             Supply an additional executor to process merge jobs independently.
 605:         def _processwith(pool, mergepool):
 607:                 {pool.submit(function, item) for item in items}, refresh=2
 630:             return _processwith(pool=self.pool, mergepool=self.mergepool)
 643:                 return _processwith(pool=poolinstance, mergepool=mergepoolinstance)
 727:                         items, repeat(self.client.submit(lambda x: x, self.heavy_input))
 854:             Default is ``False`` - results get merged as they finish in the main process.
 856:             Labels of the executors (from dfk.config.executors) that will process main 

In [7]:
from coffea.processor.executor import Runner
import inspect

print(inspect.getsource(Runner._work_function))

    @staticmethod
    def _work_function(
        format: str,
        xrootdtimeout: int,
        schema: schemas.BaseSchema,
        use_dataframes: bool,
        savemetrics: bool,
        item: WorkItem,
        processor_instance: Union[
            ProcessorABC, Callable[[awkward.highlevel.Array], Any]
        ],
        uproot_options: dict,
        iteritems_options: dict,
        checkpointer: CheckpointerABC,
        cache_function: Callable[[], MutableMapping],
    ) -> dict:
        if "timeout" in uproot_options:
            xrootdtimeout = uproot_options["timeout"]
        if processor_instance == "heavy":
            item, processor_instance = item
        if not isinstance(processor_instance, ProcessorABC) or not callable(
            processor_instance
        ):
            processor_instance = cloudpickle.loads(lz4f.decompress(processor_instance))

        metadata = {
            "dataset": item.dataset,
            "filename": item.filename,
            "treename":

In [8]:
print("Opening file", item.filename)

filecontext = uproot.open(...)

print("Opened file")

events = factory.events()

print("Created events")

out = processor_instance.process(events)

print("Finished processing")

NameError: name 'item' is not defined

In [3]:
import inspect
from coffea import processor

print(inspect.getfile(processor.DaskExecutor))

/usr/local/lib/python3.12/site-packages/coffea/processor/executor.py


In [3]:
out = coffea.util.load("outputs/bkg_24_ttj.coffea")

In [6]:
print(type(out))
print(out.keys())
for k in out:
    print(k)

<class 'dict'>
dict_keys(['TTJets'])
TTJets


In [7]:
sample = list(out.keys())[0]

print(type(out[sample]))
print(out[sample].keys())

<class 'dict'>
dict_keys(['cutflow', 'hists', 'counters', 'metadata'])


In [8]:
# Look at available histograms
print(out[sample]["hists"].keys())

dict_keys(['genE_n', 'genE_pt', 'genE_pt_highRange', 'genE_dxy', 'genE_dxy_lowRange', 'genE_dxy_XLowRange', 'genE_dxy_XXLowRange', 'genE0_pt', 'genE1_pt', 'genE0_pt_highRange', 'genE1_pt_highRange', 'genE_eta_phi', 'genE_parent_absPdgId', 'genE_genE_dR', 'genE_genE_dR_lowRange', 'genE_genE_dR_XLowRange', 'genE_genE_dR_XXLowRange', 'genE_genE_dEta', 'genE_genE_dPhi', 'genE_genE_dPt', 'genE_genE_pt', 'electron_genE_dR', 'photon_genE_dR', 'genE_matched_electron_dxy', 'genE_matched_electron_dxy_lowRange', 'genE_matched_electron_dxy_XLowRange', 'genE_matched_lj_electron_dxy', 'genE_matched_lj_electron_dxy_lowRange', 'genE_matched_lj_electron_dxy_XLowRange', 'genE0_dxy', 'genE1_dxy', 'genE0_dxy_lowRange', 'genE1_dxy_lowRange', 'genE_matched_electron_status', 'genMu_n', 'genMu_pt', 'genMu_pt_highRange', 'genMu_dxy', 'genMu_dxy_lowRange', 'genMu_dxy_XLowRange', 'genMu_dxy_XXLowRange', 'genMu0_pt', 'genMu1_pt', 'genMu0_pt_highRange', 'genMu1_pt_highRange', 'genMu_eta_phi', 'genMu_parent_absPdgI

In [10]:
channels = ["baseNoLj",]
ch1 = channels[0]

In [2]:
utilities.plot(out[sample]["hists"]["lj_lj_invmass"][ch1, :], label = "sig4mu[0]")

NameError: name 'utilities' is not defined

In [1]:
h = out[sample]["hists"]["lj_eta_phi"]

print(h.axes)

NameError: name 'out' is not defined

In [18]:
output = coffea.util.load(f"outputs/bkg_31.coffea")
channels = ["baseNoLj",
            "bkg_base", 
            "bkg_base_iso",
            "bkg_base_iso_disp",
            "bkg_base_iso_disp_2mu2e",
            "bkg_base_iso_disp_2mu2e_dphi",
            "bkg_base_iso_disp_4mu",
            "bkg_base_iso_disp_4mu_dphi",
]
ch6 = channels[5]

In [22]:
signal = output["2Mu2E_500GeV_0p25GeV_0p004mm"]["cutflow"][ch6].rows["LJ-LJ dPhi > 2"]["weighted"]
print(signal)
ttj = output["TTJets"]["cutflow"][ch6].rows["LJ-LJ dPhi > 2"]["weighted"]
print(ttj)

32.275604
2624.004


In [8]:
# check samples in output
for s in output:
    print(s)

2Mu2E_500GeV_1p2GeV_9p6mm
2Mu2E_500GeV_1p2GeV_1p9mm
2Mu2E_200GeV_5p0GeV_100p0mm
2Mu2E_500GeV_5p0GeV_80p0mm
2Mu2E_800GeV_5p0GeV_50p0mm
2Mu2E_1000GeV_5p0GeV_40p0mm
2Mu2E_500GeV_0p25GeV_4p0mm
2Mu2E_500GeV_1p2GeV_19p0mm
2Mu2E_500GeV_1p2GeV_0p019mm
2Mu2E_500GeV_1p2GeV_0p19mm


In [2]:
output = coffea.util.load(f"outputs/bkg_24_2mu.coffea")

In [ ]:
print(output[sample]["cutflow"].keys())

In [6]:
#check channels in output
h = output["2Mu2E_500GeV_1p2GeV_9p6mm"]["hists"]["lj0_pt"]
print(h.axes)

(StrCategory(['baseNoLj', 'bkg_iso', 'bkg_iso_disp', 'bkg_iso_disp_2lj', 'bkg_iso_disp_2lj_2mu2e', 'bkg_iso_disp_2lj_4mu', 'bkg_iso_disp_2lj_2mu2e_dphi', 'bkg_iso_disp_2lj_4mu_dphi'], name='channel'), Regular(100, 0, 400, name='lj0_pt', label='Leading lepton jet pT [GeV]'))


In [4]:
# output

{'2Mu2E_500GeV_1p2GeV_9p6mm': {'cutflow': {'baseNoLj': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8e0f7e6c0>,
   'bkg_iso': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dd5346e0>,
   'bkg_iso_disp': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd98890>,
   'bkg_iso_disp_2lj': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd989b0>,
   'bkg_iso_disp_2lj_2mu2e': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd98c20>,
   'bkg_iso_disp_2lj_4mu': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd98e90>,
   'bkg_iso_disp_2lj_2mu2e_dphi': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd99100>,
   'bkg_iso_disp_2lj_4mu_dphi': <sidm.tools.cutflow.SimpleCutflow at 0x7fa8dcd993d0>},
  'hists': {'genE_n': Hist(
     StrCategory(['baseNoLj', 'bkg_iso', 'bkg_iso_disp', 'bkg_iso_disp_2lj', 'bkg_iso_disp_2lj_2mu2e', 'bkg_iso_disp_2lj_4mu', 'bkg_iso_disp_2lj_2mu2e_dphi', 'bkg_iso_disp_2lj_4mu_dphi'], name='channel'),
     Regular(10, 0, 10, name='genEs_n', label='Number of genEss'),
     storage=Weight()) # Sum: Weig

In [2]:
nfiles = 1
sig2mu = [
    # "2Mu2E_500GeV_1p2GeV_0p19mm", #0.3
    # "2Mu2E_500GeV_1p2GeV_1p9mm", #30
    # "2Mu2E_200GeV_5p0GeV_100p0mm",
    "4Mu_500GeV_1p2GeV_19p0mm",
    
]

sig4mu = [
    "4Mu_500GeV_1p2GeV_19p0mm",
]
channels = ["baseNoLj", 
            "bkg_iso_disp_2lj_4mu_dphi",
            "bkg_iso_disp_2lj_4mu_dphi_ljm",
           ]

bkgdyj = [
    "DYJetsToMuMu_M10to50",
    "DYJetsToMuMu_M50",
]
bkgttj = ["TTJets"]

bkgqcd = [
    # "QCD_Pt15To20",
    # "QCD_Pt20To30",
    # "QCD_Pt30To50",
    # "QCD_Pt50To80",
    "QCD_Pt80To120",
]

bkgocd = [
    "QCD_Pt120To170",
    # "QCD_Pt170To300",
    # "QCD_Pt300To470",
    "QCD_Pt470To600",
    "QCD_Pt600To800",
    # "QCD_Pt800To1000", 
    "QCD_Pt1000",       
]


ch1 = channels[0] #base
ch2 = channels[1] # 4mu
ch3 = channels[1] # mdiff <0.1

In [3]:
runner = processor.Runner(
    executor=processor.FuturesExecutor(),                # for testing locally
    schema = llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=True,
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    unweighted_hist=True, verbose=False
)

In [4]:
# process 2mu2e

# fileset2mu = utilities.make_fileset(sig2mu, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_2mu2e_v10.yaml")
# out2mu = runner.run(fileset2mu, treename="Events", processor_instance=p)
# out2mu = out2mu["out"]
# coffea.util.save(out_2mu, "outputs/bkg_26_2mu.coffea")

In [5]:
# process 2mu2e
# fileset2mu = utilities.make_fileset(sig2mu, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_2mu2e_v10.yaml")
# out2mu = runner.run(fileset2mu, treename="Events", processor_instance=p)
# out2mu = out2mu["out"]

In [6]:
#process 4mu
fileset4mu = utilities.make_fileset(sig4mu, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_4mu_v10.yaml")
out4mu = runner.run(fileset4mu, treename="Events", processor_instance=p)
out4mu = out4mu["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


4Mu_500GeV_1p2GeV_19p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb


In [7]:
# process DYJ
filesetdyj = utilities.make_fileset(bkgdyj, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outdyj     = runner.run(filesetdyj, treename="Events", processor_instance=p)
outdyj     = outdyj["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(


DYJetsToMuMu_M10to50 is simulation. Scaling histograms or cutflows according to lumi*xs.
DYJetsToMuMu_M50 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [8]:
# Process ttj
filesetttj = utilities.make_fileset(bkgttj, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outttj     = runner.run(filesetttj, treename="Events", processor_instance=p)
outttj     = outttj["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


TTJets is simulation. Scaling histograms or cutflows according to lumi*xs.


In [9]:
# process qcd
filesetqcd    = utilities.make_fileset(bkgqcd, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd        = runner.run(filesetqcd, treename="Events", processor_instance=p)
outqcd        = outqcd["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


QCD_Pt80To120 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [10]:
# process ocd
filesetocd    = utilities.make_fileset(bkgocd, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outocd        = runner.run(filesetocd, treename="Events", processor_instance=p)
outocd        = outocd["out"]

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(
/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(
/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(


QCD_Pt120To170 is simulation. Scaling histograms or cutflows according to lumi*xs.
QCD_Pt470To600 is simulation. Scaling histograms or cutflows according to lumi*xs.
QCD_Pt600To800 is simulation. Scaling histograms or cutflows according to lumi*xs.
QCD_Pt1000 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [12]:
outall = {
    # **out2mu, 
    **out4mu,
    **outdyj,
    **outttj,
    **outqcd, 
    **outocd,
}

In [ ]:
plt.subplots(1, 1, figsize=(16, 10))
plt.subplot(1,2,1)
utilities.plot(outall[sig4mu[0]]["hists"]["lj_lj_invmass"][ch2, :], label = sig4mu[0])
# utilities.plot(outall[sig4mu[0]]["hists"]["lj_lj_invmass"][ch1, :], label = "lj2")
utilities.plot(outall[bkgdyj[0]]["hists"]["lj_lj_invmass"][ch2, :], label = "dyj")
utilities.plot(outall[bkgttj[0]]["hists"]["lj_lj_invmass"][ch2, :], label = "ttj")

plt.yscale("log")
plt.title("-")
plt.legend()

plt.subplot(1,2,2)
utilities.plot(outall[sig4mu[0]]["hists"]["lj_lj_invmass"][ch3, :], label = sig4mu[0])
# utilities.plot(outall[sig4mu[0]]["hists"]["lj_lj_invmass"][ch1, :], label = "lj2")
utilities.plot(outall[bkgdyj[0]]["hists"]["lj_lj_invmass"][ch3, :], label = "dyj")
utilities.plot(outall[bkgttj[0]]["hists"]["lj_lj_invmass"][ch3, :], label = "ttj")

plt.yscale("log")
plt.title("-")
plt.legend()

/usr/local/lib/python3.12/site-packages/mplhep/utils.py:652: RuntimeWarning: All sumw are zero!  Cannot compute meaningful error bars
  return np.abs(method_fcn(self.values(), variances) - self.values())
/usr/local/lib/python3.12/site-packages/mplhep/utils.py:652: RuntimeWarning: All sumw are zero!  Cannot compute meaningful error bars
  return np.abs(method_fcn(self.values(), variances) - self.values())
/usr/local/lib/python3.12/site-packages/mplhep/utils.py:652: RuntimeWarning: All sumw are zero!  Cannot compute meaningful error bars
  return np.abs(method_fcn(self.values(), variances) - self.values())
/usr/local/lib/python3.12/site-packages/mplhep/utils.py:652: RuntimeWarning: All sumw are zero!  Cannot compute meaningful error bars
  return np.abs(method_fcn(self.values(), variances) - self.values())


In [ ]:
plt.subplots(1, 2, figsize=(32, 10))
plt.subplot(1,2,1)
utilities.plot(out4mu[sig4mu[0]]["hists"]["mulj1Mass"][ch1, :], label = "lj1")
utilities.plot(out4mu[sig4mu[0]]["hists"]["mulj2Mass"][ch1, :], label = "lj2")
plt.yscale("log")
plt.title(sig4mu[0])
plt.legend()

plt.subplot(1, 2, 2)
utilities.plot(out4mu[sig4mu[0]]["hists"]["mulj1Mass"][ch2, :], label = "lj1")
utilities.plot(out4mu[sig4mu[0]]["hists"]["mulj2Mass"][ch2, :], label = "lj2")
plt.yscale("log")
plt.title(sig4mu[0])
plt.legend()
# plt.savefig(f"clean_plots/fin_mass.pdf", bbox_inches="tight", dpi=300)

In [ ]:
plt.subplots(1, 2, figsize=(32, 10))
plt.subplot(1,2,1)
utilities.plot(out4mu[sig4mu[0]]["hists"]["muljs_Massdiff"][ch1, ::2j], label = "No cut")
plt.yscale("log")
plt.title(sig4mu[0])
plt.legend()

plt.subplot(1, 2, 2)
utilities.plot(out4mu[sig4mu[0]]["hists"]["muljs_Massdiff"][ch2, ::2j], label = r"$\Delta(m1, m2) < 0.1$")
plt.yscale("log")
plt.title(sig4mu[0])
plt.legend()
# plt.savefig(f"clean_plots/fin_mass_diff.pdf", bbox_inches="tight", dpi=300)

In [ ]:
plt.subplots(1, 1, figsize=(16, 10))
# plt.subplot(1,2,1)
utilities.plot(out4mu[sig4mu[0]]["hists"]["lj_lj_invmass"][ch1, ::2j], label = "No cut")
utilities.plot(out4mu[sig4mu[0]]["hists"]["lj_lj_invmass"][ch2, ::2j], label = "4mu")
utilities.plot(out4mu[sig4mu[0]]["hists"]["lj_lj_invmass"][ch3, ::2j], label = r"$\Delta(m1, m2) < 0.1$")
plt.yscale("log")
plt.title(sig4mu[0])
plt.legend()

# plt.subplots(1, 2, figsize=(32, 10))
# plt.subplot(1,2,1)
# utilities.plot(out4mu[sig4mu[0]]["hists"]["lj_lj_invmass"][ch2, ::2j], label = "No cut")
# plt.yscale("log")
# plt.title(sig4mu[0])
# plt.legend()

# plt.subplot(1, 2, 2)
# utilities.plot(out4mu[sig4mu[0]]["hists"]["muljs_Massdiff"][ch2, ::2j], label = r"$\Delta(m1, m2) < 0.1$")
# plt.yscale("log")
# plt.title(sig4mu[0])
# plt.legend()
# plt.savefig(f"clean_plots/fin_mass_diff.pdf", bbox_inches="tight", dpi=300)

In [ ]:
plt.subplots(1, 1, figsize=(16, 10))
# utilities.plot(out2mu[sig2mu[0]]["hists"]["muon_muon_dR"][ch1, ::3j], label = "dR")
utilities.plot(out2mu[sig2mu[0]]["hists"]["muon_muon_dR2"][ch1, ::3j], label = "dR2")

In [ ]:
plt.subplots(1, 1, figsize=(16, 10))
# utilities.plot(out2mu[sig2mu[0]]["hists"]["muon_muon_dR"][ch1, ::3j], label = "dR")
utilities.plot(out2mu[sig2mu[0]]["hists"]["muon_genMu_matched_dR"][ch1, ::3j], label = "dR2")

In [ ]:
plt.subplots(1, 1, figsize=(16, 10))
utilities.plot(out2mu[sig2mu[0]]["hists"]["muon_pt_resolution"][ch1, :], label = "PF")
utilities.plot(out2mu[sig2mu[0]]["hists"]["dsaMuon_pt_resolution"][ch1, :], label = "DSA")
plt.legend()

In [ ]:
h = out2mu[sig2mu[0]]["hists"]["lj_eta_phi"]
# print(h.axes)

plt.subplots(1, 1, figsize=(16, 10))
utilities.plot(out2mu[sig2mu[0]]["hists"]["lj_eta_phi"][channels[0], ::2j, :].project("ljs_eta"), density=False)

plt.subplots(1, 1, figsize=(16, 10))
utilities.plot(out2mu[sig2mu[0]]["hists"]["lj_eta_phi"][channels[0], ::2j, :].project("ljs_phi"), density=False)